In [2]:
print("Hello")

Hello


In [3]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout


In [ ]:


!pip install pyarrow fastparquet

In [4]:
import os

print(os.listdir('/content'))

['.config', '.part-00000-6d0ca5a2-cc67-4b28-840a-873eb9d50a10-c000.snappy.parquet.crc', 'part-00000-6d0ca5a2-cc67-4b28-840a-873eb9d50a10-c000.snappy.parquet', '.part-00000-2f3b7654-f2e6-47a7-8076-809f7c7a1346-c000.snappy.parquet.crc', 'part-00000-2f3b7654-f2e6-47a7-8076-809f7c7a1346-c000.snappy.parquet', '._SUCCESS.crc', '.part-00000-c0af9417-3f15-45f5-b47f-30d32805745c-c000.snappy.parquet.crc', 'part-00000-c0af9417-3f15-45f5-b47f-30d32805745c-c000.snappy.parquet', 'sample_data']


In [5]:
import pandas as pd

df = pd.read_parquet(
    '/content/part-00000-2f3b7654-f2e6-47a7-8076-809f7c7a1346-c000.snappy.parquet'
)

# Check dataset
print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (1000000, 17)


,Page,Date,value,Moving_avg,Moving_std,lag_1,lag_2,trend,deviation,z_score,percentile,state_label,Page_index,year,month,day,features
0,10._August_de.wikipedia.org_desktop_all-agents,2016-12-05,24.0,24.67,0.58,25.0,25.0,-1.0,-0.67,-1.16,0.005566,Normal,79628.0,2016,12,5,"{'type': 1, 'size': None, 'indices': None, 'va..."
1,10._August_de.wikipedia.org_desktop_all-agents,2016-12-24,15.0,17.00,1.73,18.0,18.0,-3.0,-2.00,-1.16,0.005566,Normal,79628.0,2016,12,24,"{'type': 1, 'size': None, 'indices': None, 'va..."
2,10._August_de.wikipedia.org_desktop_all-agents,2015-07-21,37.0,60.00,19.92,71.0,72.0,-34.0,-23.00,-1.15,0.009276,Normal,79628.0,2015,7,21,"{'type': 1, 'size': None, 'indices': None, 'va..."
3,10._August_de.wikipedia.org_desktop_all-agents,2015-08-17,97.0,133.00,31.19,152.0,150.0,-55.0,-36.00,-1.15,0.009276,Normal,79628.0,2015,8,17,"{'type': 1, 'size': None, 'indices': None, 'va..."
4,10._August_de.wikipedia.org_desktop_all-agents,2015-09-02,118.0,125.00,6.08,128.0,129.0,-10.0,-7.00,-1.15,0.009276,Normal,79628.0,2015,9,2,"{'type': 1, 'size': None, 'indices': None, 'va..."


In [ ]:


!pip install tensorflow scikit-learn pyarrow fastparquet

In [6]:


# Remove unnecessary columns
drop_cols = ['Date', 'Page']

# Drop features column if exists
if 'features' in df.columns:
    drop_cols.append('features')

# Drop only columns that exist
df = df.drop(columns=[col for col in drop_cols if col in df.columns])

# Check cleaned data
print("Cleaned Shape:", df.shape)
df.head()

Cleaned Shape: (1000000, 14)


,value,Moving_avg,Moving_std,lag_1,lag_2,trend,deviation,z_score,percentile,state_label,Page_index,year,month,day
0,24.0,24.67,0.58,25.0,25.0,-1.0,-0.67,-1.16,0.005566,Normal,79628.0,2016,12,5
1,15.0,17.00,1.73,18.0,18.0,-3.0,-2.00,-1.16,0.005566,Normal,79628.0,2016,12,24
2,37.0,60.00,19.92,71.0,72.0,-34.0,-23.00,-1.15,0.009276,Normal,79628.0,2015,7,21
3,97.0,133.00,31.19,152.0,150.0,-55.0,-36.00,-1.15,0.009276,Normal,79628.0,2015,8,17
4,118.0,125.00,6.08,128.0,129.0,-10.0,-7.00,-1.15,0.009276,Normal,79628.0,2015,9,2


In [11]:
from sklearn.preprocessing import LabelEncoder

# Separate features and target
X = df.drop(columns=['state_label'])
y = df['state_label']

# Convert features only
X = X.apply(pd.to_numeric, errors='coerce')

# Fill missing values
X = X.fillna(X.mean())

# Encode target labels
encoder = LabelEncoder()
y = encoder.fit_transform(y)

# Convert datatype
X = X.astype('float32')

print("Features Shape:", X.shape)
print("Labels Shape:", y.shape)

Features Shape: (1000000, 13)
Labels Shape: (1000000,)


In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(800000, 13)
(200000, 13)


In [13]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [14]:
X_train_gru = np.reshape(
    X_train_scaled,
    (X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
)

X_test_gru = np.reshape(
    X_test_scaled,
    (X_test_scaled.shape[0], X_test_scaled.shape[1], 1)
)

print("GRU Train Shape:", X_train_gru.shape)
print("GRU Test Shape :", X_test_gru.shape)


GRU Train Shape: (800000, 13, 1)
GRU Test Shape : (200000, 13, 1)


In [15]:

import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
df = pd.read_parquet(
    '/content/part-00000-2f3b7654-f2e6-47a7-8076-809f7c7a1346-c000.snappy.parquet'
)

print("Original Dataset Shape:", df.shape)


required_columns = [
    'value',
    'Moving_avg',
    'Moving_std',
    'lag_1',
    'lag_2',
    'trend',
    'state_label'
]

df = df[required_columns]

print("Selected Dataset Shape:", df.shape)


feature_cols = [
    'value',
    'Moving_avg',
    'Moving_std',
    'lag_1',
    'lag_2',
    'trend'
]

for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')


df = df.dropna(subset=feature_cols)

label_encoder = LabelEncoder()
df['state_label'] = label_encoder.fit_transform(df['state_label'])

print("After Cleaning Shape:", df.shape)


# splitting the features and labeling

X = df[feature_cols].astype('float32')
y = df['state_label'].astype('float32')

print("Final Features Shape:", X.shape)
print("Final Labels Shape:", y.shape)

#training and test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False,
    random_state=42
)

print("Train Shape:", X_train.shape)
print("Test Shape :", X_test.shape)

# scaling features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# Reshaping for GRU
X_train_gru = X_train_scaled.reshape(
    X_train_scaled.shape[0],
    X_train_scaled.shape[1],
    1
)

X_test_gru = X_test_scaled.reshape(
    X_test_scaled.shape[0],
    X_test_scaled.shape[1],
    1
)

print("GRU Train Shape:", X_train_gru.shape)
print("GRU Test Shape :", X_test_gru.shape)

# classes number classification
num_classes = len(np.unique(y))


# STEP 11: building model
if num_classes == 2:
    # Binary classification
    output_units = 1
    output_activation = 'sigmoid'
    loss_function = 'binary_crossentropy'
else:
    # Multi-class classification
    output_units = num_classes
    output_activation = 'softmax'
    loss_function = 'sparse_categorical_crossentropy'

gru_model = Sequential([
    GRU(
        units=64,
        return_sequences=False,
        input_shape=(X_train_gru.shape[1], 1)
    ),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(output_units, activation=output_activation)
])

gru_model.compile(
    optimizer='adam',
    loss=loss_function,
    metrics=['accuracy']
)

gru_model.summary()


#  Training model

start_time = time.time()

history = gru_model.fit(
    X_train_gru,
    y_train,
    epochs=10,
    batch_size=256,
    validation_split=0.1,
    verbose=1
)

training_time = time.time() - start_time


# Prediction

y_pred_prob = gru_model.predict(X_test_gru)

if num_classes == 2:
    y_pred = (y_pred_prob > 0.5).astype(int).flatten()
else:
    y_pred = np.argmax(y_pred_prob, axis=1)


# STEP 14: Evaluation

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')


# results
print("\n===== GRU Model Performance =====")
print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))
print("Training Time (seconds):", round(training_time, 2))


# SAVE RESULTS

results = pd.DataFrame([{
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1_Score": f1,
    "Training_Time_Seconds": training_time
}])

#results.to_csv("/content/gru_results.csv", index=False)

print("\nResults saved successfully.")

Original Dataset Shape: (1000000, 17)
Selected Dataset Shape: (1000000, 7)
After Cleaning Shape: (1000000, 7)
Final Features Shape: (1000000, 6)
Final Labels Shape: (1000000,)
Train Shape: (800000, 6)
Test Shape : (200000, 6)
GRU Train Shape: (800000, 6, 1)
GRU Test Shape : (200000, 6, 1)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 64)             │        12,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,043 (58.76 KB)

 Trainable params: 15,043 (58.76 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 26s 6ms/step - accuracy: 0.7414 - loss: 0.5714 - val_accuracy: 0.8192 - val_loss: 0.3795
Epoch 2/10
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - accuracy: 0.8432 - loss: 0.3759 - val_accuracy: 0.8863 - val_loss: 0.3011
Epoch 3/10
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 17s 6ms/step - accuracy: 0.8664 - loss: 0.3284 - val_accuracy: 0.8901 - val_loss: 0.2904
Epoch 4/10
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - accuracy: 0.8767 - loss: 0.3037 - val_accuracy: 0.8796 - val_loss: 0.2743
Epoch 5/10
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 19s 7ms/step - accuracy: 0.8825 - loss: 0.2887 - val_accuracy: 0.9009 - val_loss: 0.2481
Epoch 6/10
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 18s 6ms/step - accuracy: 0.8880 - loss: 0.2758 - val_accuracy: 0.9096 - val_loss: 0.2338
Epoch 7/10
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 20s 7ms/step - accuracy: 0.8897 - loss: 0.2685 - val_accuracy: 0.9136 - val_loss: 0.2264
Epoch 8/10
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 18s 6ms/step - accuracy: 0.8926 - loss: 0

In [ ]:
#While baseline ANN models exhibited scalability saturation and predictive degradation at larger distributed scales,
#GRU demonstrated robust distributed scalability with sustained high predictive performance across 1M–5M datasets.
#This indicates that recurrent architectures may better exploit large-scale distributed sequential data environments than
# standard feedforward networks.

In [ ]:
#The introduction of GRU substantially improved distributed deep learning effectiveness,
#suggesting that architecture selection plays a critical role in achieving both scalability and
#predictive performance in large-scale distributed systems.